# RIGHTWARE — SI-RD-03A A0 v0.7 | Pesquisa global e controlo de escopo

**Experiência sintética e isolada.** Não utiliza dados pessoais, clientes, passwords, PostgreSQL Server ou APIs de Mem0/Hindsight. **Nunca** declare desempenho de fornecedores com base apenas nos testes de protocolo.

**O que muda em relação a v0.6:** a recuperação deixa de filtrar pelo `case_id` conhecido do oráculo. Pesquisa em 29 documentos autorizados de um corpus de 49 fontes; 20 pertencem a outros escopos. São 40 perguntas (18 históricas, 16 desafios PT/EN, 4 sondagens de isolamento, 2 desconhecidas), 320 consultas de ranking PostgreSQL-WASM e 160 consultas de contagem de candidatos. A qualidade histórica do benchmark permanece limitada pela pequena amostra, reformulações relacionadas e método OR pós-hoc originado no v0.6.

**Instruções:** CPU / Internet ON para instalar npm; Run All. O próprio notebook já contém o laboratório — **não é necessário adicionar outro dataset**. O ficheiro `RIGHTWARE_SI_RD_03A_A0_V07_EVIDENCE.zip` surgirá na pasta `/kaggle/working`.

## 1. Recuperar e autenticar o pacote incorporado

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
import base64, hashlib, io, zipfile, shutil, subprocess, json, os, sys
WORKING = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
ROOT = WORKING / 'SI_RD_03A_A0_v07'

ARCHIVE_BASE64 = '...'
EXPECTED_SHA256 = '170d44e36e513239d68e6fc8e2959b9eeddd18d61bd21fa685775f0117599dab'
archive = base64.b64decode(ARCHIVE_BASE64)
assert hashlib.sha256(archive).hexdigest() == EXPECTED_SHA256, 'Embedded lab ZIP was modified'
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    names = z.namelist()
    assert names and all(name.startswith('SI_RD_03A_A0_v07/') and '..' not in Path(name).parts for name in names)
    assert all(not name.startswith('/') for name in names)
    z.extractall(WORKING)
assert ROOT.is_dir()
print('Pacote original SHA256:',EXPECTED_SHA256)
print('Laboratório:',ROOT)
print('Node:',subprocess.check_output(['node','--version'],text=True).strip())
print('Python:',sys.version.split()[0])


## 2. Verificações locais do contrato (não é benchmark do motor)

In [ ]:
def run_checked(command, env=None, timeout=180):
    print('CMD:', ' '.join(command))
    result=subprocess.run(command,cwd=ROOT,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(result.stdout[-10000:])
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {command}')
    return result
run_checked(['python3','src/generate_fixture_v07.py'])
run_checked(['python3','-m','unittest','discover','-s','tests','-v'])
run_checked(['node','--check','src/run_retrieval_v07.mjs'])


## 3. Instalar PGlite em ambiente descartável (sem bases remotas)

In [ ]:
import re
node_version=subprocess.check_output(['node','--version'],text=True).strip()
assert int(re.match(r'v(\d+)', node_version).group(1)) >= 20, 'Node.js 20+ obrigatório'
run_checked(['npm','install','--ignore-scripts','--no-audit','--no-fund','--save-exact'],timeout=240)
run_checked(['npm','ls','@electric-sql/pglite','--depth=0'])
lock=json.loads((ROOT/'package-lock.json').read_text())
assert lock['packages']['node_modules/@electric-sql/pglite']['version']=='0.5.8'
assert lock['packages']['node_modules/@electric-sql/pglite']['integrity'].startswith('sha512-')
print('PGlite 0.5.8 pin + npm integrity verificados')


## 4. Executar pesquisa PostgreSQL-WASM global (só dados sintéticos)

In [ ]:
env=os.environ.copy()
env['RW_SI_A0_ISOLATED_CONFIRMED']='YES_SYNTHETIC_ONLY'
run_checked(['node','src/run_retrieval_v07.mjs'],env=env,timeout=240)
raw_path=ROOT/'results/A0_V07_RAW_RETRIEVAL.json'
raw=json.loads(raw_path.read_text())
assert raw['ranked_query_count']==320 and raw['candidate_count_sql_executions']==160
assert raw['documents']==49 and raw['queries']==40
print('Version detectada:',raw['observed_postgresql_version'])
print('Recuperações reais:',raw['ranked_query_count'])


## 5. Abrir os oráculos só depois da recuperação e calcular scores

In [ ]:
run_checked(['python3','src/score_v07.py'])
score_path=ROOT/'results/A0_V07_SCORECARD.json'
score=json.loads(score_path.read_text())
assert score['scope_violations']==0 and not score['provider_selection_approved']
import pandas as pd
records=[]
for r in score['runs']:
    group=r['breakdown']['new_challenge_answerable']
    historical=r['breakdown']['historical_unambiguous']
    records.append({'Dicionário':r['dictionary'],'Método':r['method'],'Top K':r['k'],
        'Desafio v0.7':f"{group['hit_count']}/{group['n_relevant']}",
        'Histórico sem ambiguidade':f"{historical['hit_count']}/{historical['n_relevant']}",
        'p50 latência WASM (ms)':r['latency_ms_p50']})
display(pd.DataFrame(records))
print('Isolamento de fontes não autorizadas: PASS no teste sintético; não implica segurança de produção')
print('Abstenção, temporalidade e autoridade: NOT_EVALUATED')


## 6. Criar ficheiro de evidências para revisão independente

In [ ]:
from datetime import datetime, timezone
items=[
 'README.md','package.json','package-lock.json',
 'src/generate_fixture_v07.py','src/run_retrieval_v07.mjs','src/score_v07.py',
 'fixtures/A0_PGLITE_DATASET_V05.json','fixtures/A0_V07_PUBLIC_CORPUS.json','fixtures/A0_V07_PUBLIC_CORPUS.json.sha256',
 'fixtures/A0_V07_SEALED_ORACLES.json','fixtures/A0_V07_SEALED_ORACLES.json.sha256',
 'docs/PROTOCOL_A0_V07.md','results/A0_V07_RAW_RETRIEVAL.json','results/A0_V07_SCORECARD.json',
]
manifest={'schema':'SI_RD_03A_A0_V07_MANIFEST','archive_created_utc':datetime.now(timezone.utc).isoformat(),
          'embedded_lab_sha256':EXPECTED_SHA256,
          'status':'PGLITE_WASM_EXECUTED__NO_MEM0_HINDSIGHT_POSTGRES_SERVER', 'files':{}}
for name in items:
    p=ROOT/name
    assert p.is_file(),f'Evidence missing: {name}'
    manifest['files'][name]={'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size}
manifest_bytes=(json.dumps(manifest,ensure_ascii=False,indent=2,sort_keys=True)+'\n').encode()
evidence=WORKING/'RIGHTWARE_SI_RD_03A_A0_V07_EVIDENCE.zip'
with zipfile.ZipFile(evidence,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in items:z.write(ROOT/name,arcname=name)
    z.writestr('MANIFEST_V07.json',manifest_bytes)
with zipfile.ZipFile(evidence) as z:
    assert z.testzip() is None
print('Evidence ZIP:', evidence)
print('Size bytes:', evidence.stat().st_size)
print('SHA256:', hashlib.sha256(evidence.read_bytes()).hexdigest())
print('Envie o ZIP de evidências ao chat de investigação SI-RD-03A.')
os.chdir(WORKING)
display(FileLink(evidence.name))


## Limites de interpretação

Uma consulta que recupera um documento não prova que a resposta, a data, o proprietário ou a aprovação estejam correctos. O filtragem por âmbito nesta experiência é uma simulação com identidade fixa, não uma validação de autorização empresarial. Os casos históricos MEM-02/MEM-03 são ambíguos no corpus global e não constituem prova de raciocínio temporal.

A consulta OR foi escolhida **depois de observar** os resultados v0.6; o protocolo v0.7 fica congelado antes de executar, mas **não** é um benchmark cego. Não comparar directamente percentagens v0.6 e v0.7 como se tivessem a mesma população. Para descarregar o ZIP, se o FileLink apresentar 404, usar Kaggle → Output/Files → `/kaggle/working` após gravar a versão.